# Prepare and Deploy Model for Inferentia2 Inference

This notebook:
1. **Merges** the LoRA adapter with the base model (SageMaker job on trn1.2xlarge)
2. **Pre-compiles** the merged model for Inferentia2 (eliminates lazy compilation)
3. **Deploys** the pre-compiled model to an `ml.inf2.xlarge` endpoint

Once complete, the endpoint is ready for instant inference (no compilation wait).

**Estimated time:** ~20 minutes (merge ~5 min + export ~10 min + deploy ~5 min)

**Prerequisite:** Run [01_fine-tuning.ipynb](01_fine-tuning.ipynb) first.

**Notebook kernel:** Python 3 (ipykernel) on `ml.t3.large`

---
## 1. Setup

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

In [ ]:
import os
import json
import boto3
import sagemaker
from config import MODEL_ID
from sagemaker.core.helper.session_helper import Session, get_execution_role

boto_session = boto3.Session()
sagemaker_session = Session(boto_session)
role = get_execution_role()

bucket = sagemaker_session.default_bucket()
region = boto_session.region_name

S3_PREFIX = "lab8-trainium-inferentia"
s3 = boto3.client('s3', region_name=region)
sm_client = boto3.client('sagemaker', region_name=region)

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")

---
## 2. Merge LoRA Adapter (~5 min)

The training job saved a LoRA adapter in Neuron distributed format.
This step consolidates it and merges with the base model.

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode, InputData
from sagemaker.core.shapes import OutputDataConfig

# Find the latest completed LoRA training job
paginator = sm_client.get_paginator("list_training_jobs")
training_job_name = None
for page in paginator.paginate(
    StatusEquals="Completed",
    SortBy="CreationTime",
    SortOrder="Descending",
    PaginationConfig={"MaxItems": 20},
):
    for job in page["TrainingJobSummaries"]:
        if "lora-trn1" in job["TrainingJobName"]:
            training_job_name = job["TrainingJobName"]
            break
    if training_job_name:
        break

if not training_job_name:
    raise ValueError("No completed LoRA training job found. Run 01_fine-tuning.ipynb first.")

adapter_s3_uri = f"s3://{bucket}/{S3_PREFIX}/output/{training_job_name}/output/model.tar.gz"
print(f"Training job: {training_job_name}")
print(f"Adapter: {adapter_s3_uri}")

In [ ]:
# Check if merged model already exists
merged_s3_key = f"{S3_PREFIX}/models/merged/{training_job_name}/model.tar.gz"
merged_s3_uri = None

# Search for existing merge job output
response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{S3_PREFIX}/models/merged/{training_job_name}/", MaxKeys=10)
for obj in response.get('Contents', []):
    if obj['Key'].endswith('model.tar.gz'):
        merged_s3_uri = f"s3://{bucket}/{obj['Key']}"
        print(f"Merged model already exists: {merged_s3_uri}")
        print("Skipping merge step.")
        break

if merged_s3_uri is None:
    TRAINING_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-training-neuronx:2.8.0-transformers4.55.4-neuronx-py310-sdk2.26.0-ubuntu22.04"
    merged_output_uri = f"s3://{bucket}/{S3_PREFIX}/models/merged/{training_job_name}"

    merger = ModelTrainer(
        training_image=TRAINING_IMAGE,
        source_code=SourceCode(
            source_dir="src",
            entry_script="consolidate_and_merge.py",
        ),
        compute=Compute(
            instance_type="ml.trn1.2xlarge",
            instance_count=1,
            volume_size_in_gb=100,
        ),
        role=role,
        base_job_name="neuron-merge",
        output_data_config=OutputDataConfig(s3_output_path=merged_output_uri),
        hyperparameters={"model_id": MODEL_ID},
    )

    print("Launching merge job (~5 min)...")
    merger.train(
        input_data_config=[InputData(channel_name="adapter", data_source=adapter_s3_uri)],
        wait=True,
    )

    merge_job_name = merger._latest_training_job.training_job_name
    merged_s3_uri = f"{merged_output_uri}/{merge_job_name}/output/model.tar.gz"
    print(f"Merged model: {merged_s3_uri}")

---
## 3. Pre-Compile for Inferentia2 (~10 min)

Compile the merged model to Neuron format so the inference endpoint
starts instantly without lazy compilation on first request.

In [ ]:
# Check if neuron-compiled model already exists
compiled_s3_key = f"{S3_PREFIX}/models/compiled/{training_job_name}/"
compiled_s3_uri = None

response = s3.list_objects_v2(Bucket=bucket, Prefix=compiled_s3_key, MaxKeys=10)
for obj in response.get('Contents', []):
    if obj['Key'].endswith('model.tar.gz'):
        compiled_s3_uri = f"s3://{bucket}/{obj['Key']}"
        print(f"Compiled model already exists: {compiled_s3_uri}")
        print("Skipping compilation step.")
        break

if compiled_s3_uri is None:
    TRAINING_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-training-neuronx:2.8.0-transformers4.55.4-neuronx-py310-sdk2.26.0-ubuntu22.04"
    compiled_output_uri = f"s3://{bucket}/{S3_PREFIX}/models/compiled/{training_job_name}"

    exporter = ModelTrainer(
        training_image=TRAINING_IMAGE,
        source_code=SourceCode(
            source_dir="src",
            entry_script="export_for_inference.py",
        ),
        compute=Compute(
            instance_type="ml.trn1.2xlarge",
            instance_count=1,
            volume_size_in_gb=100,
        ),
        role=role,
        base_job_name="neuron-export",
        output_data_config=OutputDataConfig(s3_output_path=compiled_output_uri),
        hyperparameters={
            "model_id": MODEL_ID,
            "batch_size": 1,
            "sequence_length": 512,
            "num_cores": 2,
            "auto_cast_type": "fp16",
        },
    )

    print("Launching Neuron export job (~10 min)...")
    print("This compiles the model for Inferentia2 so inference starts instantly.")
    exporter.train(
        input_data_config=[InputData(channel_name="model", data_source=merged_s3_uri)],
        wait=True,
    )

    export_job_name = exporter._latest_training_job.training_job_name
    compiled_s3_uri = f"{compiled_output_uri}/{export_job_name}/output/model.tar.gz"
    print(f"Compiled model: {compiled_s3_uri}")

---
## 4. Deploy to Inferentia2 Endpoint (~5 min)

Deploy the pre-compiled model. No lazy compilation needed — endpoint is
ready for inference immediately after reaching InService.

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
from time import gmtime, strftime

INFERENCE_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-inference-neuronx:2.8.0-transformers4.55.4-neuronx-py310-sdk2.26.0-ubuntu22.04"

timestamp = strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model_name = f"qwen3-inf2-{timestamp}"
endpoint_name = f"qwen3-inf2-{timestamp}"

sm_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=INFERENCE_IMAGE,
        model_data_url=compiled_s3_uri,
        environment={
            "HF_TASK": "text-generation",
            "HF_NUM_CORES": "2",
            "HF_OPTIMUM_SEQUENCE_LENGTH": "512",
            "HF_BATCH_SIZE": "1",
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1",
        },
    ),
    execution_role_arn=role,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[ProductionVariant(
        variant_name="AllTraffic",
        model_name=model_name,
        initial_instance_count=1,
        instance_type="ml.inf2.xlarge",
        model_data_download_timeout_in_seconds=600,
        container_startup_health_check_timeout_in_seconds=600,
    )],
)

sm_client.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)

print(f"Creating endpoint '{endpoint_name}'...")
print("With pre-compiled model, this should take ~5 minutes (no lazy compilation).")

waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=endpoint_name, WaiterConfig={"Delay": 30, "MaxAttempts": 30})
print(f"\nEndpoint ready: {endpoint_name}")

/---
## Summary

| Step | What | Time |
|------|------|------|
| Merge | Consolidate LoRA adapter + merge with base model | ~5 min |
| Export | Compile model for Inferentia2 (Neuron format) | ~10 min |
| Deploy | Create SageMaker endpoint | ~5 min |

All steps are cached in S3 — re-running skips completed steps.

**Next:** Open [03_inference.ipynb](03_inference.ipynb) to send prompts to the endpoint.